In [2]:
import json
import re
import time
import os
import pandas as pd
from openai import OpenAI
from dotenv import load_dotenv

In [3]:
load_dotenv()
API_KEY = os.getenv("DEEPSEEK_API_KEY")

INPUT_FILE   = r"C:\Users\USER\Downloads\qwen3_6_results_with_pd.csv"
OUTPUT_0SHOT = r"C:\Users\USER\Downloads\qwen_judge_0shot.csv"
OUTPUT_1SHOT = r"C:\Users\USER\Downloads\qwen_judge_1shot.csv"
MODEL = "deepseek-v4-pro"

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.deepseek.com"
)

SYSTEM_PROMPT = """You are a clinical diagnostic auditor.
Evaluate the accuracy of the Model's extracted diagnosis against the Ground Truth.
Score D1 DIAGNOSIS ACCURACY (0-3).

3: Correct match (exact or clinically equivalent synonym).
2: Correct organ system/category, but wrong specificity.
1: Correct diagnosis appears only in differential/minor mention.
0: Incorrect or missing entirely.

Return ONLY JSON:
{
  "d1_accuracy": <0-3>,
  "model_dx": "<extracted model diagnosis>",
  "reasoning": "<one sentence explanation>"
}"""

def judge_accuracy(row):
    prompt = f"Model Diagnosis: {row['predicted_pd']}\nGround Truth: {row['LONG_TITLE']}"
    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": prompt}
            ],
            extra_body={"thinking": {"type": "enabled"}},
            max_tokens=3072,
            temperature=0.0
        )
        content = response.choices[0].message.content
        if not content or not content.strip():
            raise ValueError("Empty response from API")
        clean_content = re.sub(r"```json|```", "", content).strip()
        return json.loads(clean_content)
    except Exception as e:
        return {
            "d1_accuracy": -1,
            "model_dx": "",
            "reasoning": str(e)
        }

In [4]:
MIMIC_FIXED = r"C:\Users\USER\Downloads\augmented_data_mimic_fixed.csv"

df = pd.read_csv(INPUT_FILE)

# Replace wrong ground truths — neweraugmented_mimic.csv had incorrect ICD codes for most HADMs
mimic_fixed = (
    pd.read_csv(MIMIC_FIXED)[['HADM_ID', 'ground_truth', 'SHORT_TITLE', 'LONG_TITLE']]
    .drop_duplicates('HADM_ID')
)
df = df.drop(columns=[c for c in ['ground_truth', 'SHORT_TITLE', 'LONG_TITLE'] if c in df.columns])
df = df.merge(mimic_fixed, on='HADM_ID', how='left')

print(f"Loaded {len(df)} rows | ground truth source: augmented_data_mimic_fixed.csv")
print(df[['HADM_ID', 'LONG_TITLE']].drop_duplicates('HADM_ID').to_string(index=False))

df_baseline = (
    df[df['condition'].isin(['0-shot', '1-shot'])]
    .drop_duplicates(subset=['HADM_ID', 'condition'], keep='first')
    .copy()
)

n_0shot = (df_baseline['condition'] == '0-shot').sum()
n_1shot = (df_baseline['condition'] == '1-shot').sum()
print(f"\nProcessing {len(df_baseline)} baseline rows: {n_0shot} x 0-shot, {n_1shot} x 1-shot")

results = []
for i, (_, row) in enumerate(df_baseline.iterrows()):
    condition = row['condition']
    print(f"[{i+1}/{len(df_baseline)}] [{condition}] HADM {row['HADM_ID']}...")

    evaluation = judge_accuracy(row)
    print(evaluation)

    combined = {**row.to_dict(), **evaluation}
    results.append(combined)

    time.sleep(0.5)

results_df = pd.DataFrame(results)

df_0shot = results_df[results_df['condition'] == '0-shot']
df_1shot = results_df[results_df['condition'] == '1-shot']

df_0shot.to_csv(OUTPUT_0SHOT, index=False)
df_1shot.to_csv(OUTPUT_1SHOT, index=False)

print(f"\n--- Results ---")
print(f"0-shot  ({len(df_0shot)} rows) → {OUTPUT_0SHOT}  |  avg d1: {df_0shot['d1_accuracy'].mean():.2f}")
print(f"1-shot  ({len(df_1shot)} rows) → {OUTPUT_1SHOT}  |  avg d1: {df_1shot['d1_accuracy'].mean():.2f}")

Loaded 50 rows | ground truth source: augmented_data_mimic_fixed.csv
 HADM_ID                                                       LONG_TITLE
  101908          Other pulmonary insufficiency, not elsewhere classified
  128089                                           Unspecified septicemia
  130788 Other diseases of trachea and bronchus, not elsewhere classified
  146302                                         Other diseases of spleen
  166328                                    Other postoperative infection
  176830                Hemorrhage of gastrointestinal tract, unspecified
  181521                            Congestive heart failure, unspecified
  186753               Coronary atherosclerosis of native coronary artery
  194302                     Other colostomy and enterostomy complication
  197345   Secondary malignant neoplasm of retroperitoneum and peritoneum

Processing 20 baseline rows: 10 x 0-shot, 10 x 1-shot
[1/20] [0-shot] HADM 101908...
{'d1_accuracy': 1, 'model_dx': 